In [78]:
import pandas as pd

# read data
df = pd.read_csv("../data/dirty/ILO/EAR_4MTH_SEX_CUR_NB_A-20250408T0015.csv")

# regex
df["sex"] = df["sex.label"].str.extract(r"Sex: (\w)")[0]
df["ISO"] = df["note_indicator.label"].str.extract(r"Currency: (\w{3})")[0]
df["currency"] = df["classif1.label"].str.extract(r"Currency: (.+)")[0]
df["local_currency_code"] = (
    df["note_indicator.label"]
    .str.extract(r"Currency:\s*\w{3}\s*-\s*[^()]*\(\s*([A-Z]{3})\s*\)")[0]
    .str.strip()
)
df["break"] = df["note_indicator.label"].str.contains("Break in series", na=False)
# remove irrelevant data
df.drop(
    columns=[
        "obs_status.label",
        "note_classif.label",
        "note_source.label",
        "indicator.label",
        "note_indicator.label",
        "source.label",
        "sex.label",
        "classif1.label",
    ],
    inplace=True,
)

# rename columns
df.rename(
    columns={
        "ref_area.label": "country",
        "time": "year",
        "obs_value": "wage",
        "ISO": "ISO",
    },
    inplace=True,
)

df["year"] = pd.to_numeric(df["year"], errors="coerce")

# index max year per country and sex
idx = df.groupby(["country", "sex", "currency"])["year"].idxmax()
# df = df.loc[idx].reset_index(drop=True)

# last_break = df[df["break"]].groupby("ISO")["year"].max().rename("last_break")
# df = df.merge(last_break, on="ISO", how="left")
# df["last_break"] = pd.to_numeric(df["last_break"], errors="coerce")

map = {"U.S. dollars": "USD", "2021 PPP $": "PPP", "Local currency": "Local"}
df["currency"] = df["currency"].replace(map)

df.to_csv("../data/clean/ILO.csv")

In [79]:
df_ratio = df.pivot_table(
    index=["ISO", "year", "country", "currency", "local_currency_code"],
    columns="sex",
    values="wage",
    aggfunc="first",
).reset_index()

break_info = df[
    [
        "ISO",
        "year",
        "break",
        # "last_break",
    ]
].drop_duplicates(subset=["ISO", "year"])

df_ratio = df_ratio.merge(break_info, on=["ISO", "year"], how="left")

df_ratio["ratio"] = df_ratio["F"] / df_ratio["M"]

df_ratio = df_ratio[
    [
        "ISO",
        "year",
        "country",
        "T",
        "M",
        "F",
        "ratio",
        "break",
        "local_currency_code",
        # "last_break",
        "currency",
    ]
]
df_ratio

,ISO,year,country,T,M,F,ratio,break,local_currency_code,currency
0,ABW,2010,Aruba,3013.000,3338.000,2713.000,0.812762,False,AWG,Local
1,ABW,2010,Aruba,1860.382,2061.054,1675.147,0.812762,False,AWG,PPP
2,ABW,2010,Aruba,1683.240,1864.804,1515.642,0.812762,False,AWG,USD
3,AFG,2014,Afghanistan,8848.569,9135.149,5517.315,0.603966,False,AFN,Local
4,AFG,2014,Afghanistan,517.709,534.476,322.805,0.603965,False,AFN,PPP
...,...,...,...,...,...,...,...,...,...,...
8383,ZWE,2022,Zimbabwe,212.044,223.145,193.270,0.866118,False,ZWL,PPP
8384,ZWE,2022,Zimbabwe,80.864,85.097,73.704,0.866117,False,ZWL,USD
8385,ZWE,2023,Zimbabwe,406978.727,420431.180,385130.787,0.916038,False,ZWL,Local
8386,ZWE,2023,Zimbabwe,1711.902,1768.488,1620.001,0.916037,False,ZWL,PPP


In [80]:
df_ratio["female_share"] = (df_ratio["T"] - df_ratio["M"]) / (
    df_ratio["F"] - df_ratio["M"]
)
df_ratio["female_share"] = df_ratio["female_share"].clip(0, 1)
df_ratio["male_share"] = 1 - df_ratio["female_share"]

df_ratio["balanced_avg"] = (df_ratio["F"] + df_ratio["M"]) / 2
df_ratio["balanced_ratio"] = df_ratio["F"] / df_ratio["balanced_avg"]

df_ratio.to_csv("../data/clean/ILO_ratio.csv")

In [81]:
df_ratio = df_ratio.dropna(subset=["F", "M"]).reset_index(drop=True)

df_wide = df_ratio.pivot_table(
    index=[
        "ISO",
        "year",
        "country",
        "break",
        "local_currency_code",
    ],
    columns="currency",
    values=["F", "M", "T", "ratio"],
    aggfunc="first",
).reset_index()

df_wide.columns = [
    f"{metric}_{curr.lower()}" if curr else metric for metric, curr in df_wide.columns
]

shares = df_ratio[
    [
        "ISO",
        "year",
        "country",
        "female_share",
        "male_share",
        "balanced_avg",
        "balanced_ratio",
    ]
].drop_duplicates(subset=["ISO", "year", "country"])

# Merge those columns into your wide table on ISO, year, and country
df_wide = df_wide.merge(shares, on=["ISO", "year", "country"], how="left")

print(df_wide)

      ISO  year      country  break local_currency_code     F_local     F_ppp  \
0     ABW  2010        Aruba  False                 AWG    2713.000  1675.147   
1     AFG  2014  Afghanistan  False                 AFN    5517.315   322.805   
2     AFG  2020  Afghanistan   True                 AFN   10843.294   686.841   
3     AGO  2019       Angola   True                 AOA   58793.485   404.360   
4     AGO  2021       Angola  False                 AOA   69497.802   341.196   
...   ...   ...          ...    ...                 ...         ...       ...   
2408  ZWE  2011     Zimbabwe   True                 ZWL     204.576       NaN   
2409  ZWE  2014     Zimbabwe  False                 ZWL     283.740       NaN   
2410  ZWE  2021     Zimbabwe  False                 ZWL   14481.879   191.960   
2411  ZWE  2022     Zimbabwe  False                 ZWL   27635.835   193.270   
2412  ZWE  2023     Zimbabwe  False                 ZWL  385130.787  1620.001   

         F_usd     M_local 

In [82]:
from currency_converter import CurrencyConverter
from datetime import date
import numpy as np

c = CurrencyConverter()

code_map = {
    "ALK": "ALL",
}


def fill_usd_if_missing(row, metric):
    usd_col = f"{metric}_usd"
    local_col = f"{metric}_local"
    if not pd.isna(row[usd_col]):
        return row[usd_col]

    code = row["local_currency_code"]
    code = code_map.get(code, code)
    try:
        rate = c.convert(1, code, "USD", date(int(row["year"]), 1, 1))
    except Exception:
        return np.nan

    return row[local_col] * rate


# Apply in-place for Female (f), Male (m), Total (t)
for metric in ["F", "M", "T"]:
    df_wide[f"{metric}_usd"] = df_wide.apply(
        lambda r: fill_usd_if_missing(r, metric), axis=1
    )

# Recompute your USD ratio after filling
df_wide["ratio_usd"] = df_wide["F_usd"] / df_wide["M_usd"]
df_wide.to_csv("../data/clean/ILO_ratio_wide.csv")

In [83]:
url = "https://en.wikipedia.org/wiki/Historical_exchange_rates_of_Argentine_currency"
tables = pd.read_html(url)

for tbl in tables:
    if list(tbl.columns[:2]) == ["Year", "Jan"]:
        wiki_fx = tbl.copy()
        break

wiki_fx.columns = wiki_fx.columns.str.strip()
wiki_fx["Year"] = wiki_fx["Year"].astype(int)

df_rates = wiki_fx.melt(
    id_vars="Year",
    value_vars=[
        "Jan",
        "Feb",
        "Mar",
        "Apr",
        "May",
        "Jun",
        "Jul",
        "Aug",
        "Sep",
        "Oct",
        "Nov",
        "Dec",
    ],
    var_name="Month",
    value_name="ars_per_usd",
)

df_rates["Month"] = pd.to_datetime(df_rates["Month"], format="%b").dt.month
df_jan = df_rates[df_rates["Month"] == 1].copy()

# rename and compute
df_jan = df_jan.rename(columns={"Year": "year"})
df_jan["usd_per_ars"] = 1.0 / df_jan["ars_per_usd"]

# merge into wide df
df_wide = df_wide.merge(df_jan[["year", "usd_per_ars"]], on="year", how="left")
print(df_wide)

# fill in the ARS conversions
mask_ars = df_wide["local_currency_code"] == "ARS"
for metric in ["F", "M", "T"]:
    df_wide.loc[mask_ars, f"{metric}_usd"] = (
        df_wide.loc[mask_ars, f"{metric}_local"] * df_wide.loc[mask_ars, "usd_per_ars"]
    )

# recompute USD ratio
df_wide["ratio_usd"] = df_wide["F_usd"] / df_wide["M_usd"]

      ISO  year      country  break local_currency_code     F_local     F_ppp  \
0     ABW  2010        Aruba  False                 AWG    2713.000  1675.147   
1     AFG  2014  Afghanistan  False                 AFN    5517.315   322.805   
2     AFG  2020  Afghanistan   True                 AFN   10843.294   686.841   
3     AGO  2019       Angola   True                 AOA   58793.485   404.360   
4     AGO  2021       Angola  False                 AOA   69497.802   341.196   
...   ...   ...          ...    ...                 ...         ...       ...   
2408  ZWE  2011     Zimbabwe   True                 ZWL     204.576       NaN   
2409  ZWE  2014     Zimbabwe  False                 ZWL     283.740       NaN   
2410  ZWE  2021     Zimbabwe  False                 ZWL   14481.879   191.960   
2411  ZWE  2022     Zimbabwe  False                 ZWL   27635.835   193.270   
2412  ZWE  2023     Zimbabwe  False                 ZWL  385130.787  1620.001   

         F_usd     M_local 

In [84]:
aud_to_usd = {
    1950: 0.89286,
    1951: 0.89286,
    1952: 0.89286,
    1953: 0.89286,
    1954: 0.89286,
    1955: 0.89286,
    1956: 0.89286,
    1957: 0.89286,
    1958: 0.89286,
    1959: 0.89286,
    1960: 0.89286,
    1961: 0.89286,
    1962: 0.89286,
    1963: 0.89286,
    1964: 0.89286,
    1965: 0.89286,
    1966: 0.89286,
    1967: 0.89286,
    1968: 0.89286,
    1969: 0.89286,
    1970: 0.89286,
    1971: 0.88267,
    1972: 0.83870,
    1973: 0.70411,
    1974: 0.69667,
    1975: 0.76387,
    1976: 0.81828,
    1977: 0.90182,
    1978: 0.87366,
    1979: 0.89464,
    1980: 0.87824,
    1981: 0.87021,
    1982: 0.98586,
    1983: 1.11001,
    1984: 1.13952,
    1985: 1.43198,
    1986: 1.49597,
    1987: 1.42818,
    1988: 1.27991,
    # … add remaining years …
}

# 2) Map that onto df_wide:
df_wide["aud_to_usd"] = df_wide["year"].map(aud_to_usd)

# 3) Apply the conversion in place for your AUD rows:
mask = df_wide["local_currency_code"] == "AUD"
for metric in ["F", "M", "T"]:
    df_wide.loc[mask, f"{metric}_usd"] = (
        df_wide.loc[mask, f"{metric}_local"] * df_wide.loc[mask, "aud_to_usd"]
    )

# 4) Recompute your USD ratio
df_wide["ratio_usd"] = df_wide["F_usd"] / df_wide["M_usd"]
df_wide = df_wide[
    [
        "ISO",
        "year",
        "country",
        "break",
        "local_currency_code",
        "F_local",
        # "F_ppp",
        "F_usd",
        "M_local",
        # "M_ppp",
        "M_usd",
        "T_local",
        # "T_ppp",
        "T_usd",
        "ratio_local",
        # "ratio_ppp",
        "ratio_usd",
        "female_share",
        "male_share",
        "balanced_avg",
        "balanced_ratio",
    ]
]
df_wide.to_csv("../data/clean/ILO_ratio_wide.csv")

In [85]:
float_cols = df_wide.select_dtypes(include="float").columns
df_wide[float_cols] = np.trunc(df_wide[float_cols] * 1000) / 1000
df_wide = df_wide.dropna(subset=["F_usd"]).reset_index(drop=True)

df_wide.to_csv("../data/clean/ILO_ratio_wide_clean.csv")

In [87]:
low_q, high_q = df_wide["ratio_usd"].quantile([0.1, 0.9])
# print(df_wide)
# 2) Filter out the top and bottom 10%
df_mid10 = df_wide.loc[df_wide["ratio_usd"].between(low_q, high_q),].reset_index(
    drop=True
)
df_mid10.to_csv("../data/clean/ILO_wide_ratio_no_outliers.csv")

In [88]:
low_q, high_q = df_wide["ratio_usd"].quantile([0.1, 0.9])

# 2) Identify the rows you're dropping (the outliers)
mask_outliers = ~df_wide["ratio_usd"].between(low_q, high_q)
df_outliers = df_wide.loc[mask_outliers]

# 3) Display or inspect them
print(df_outliers)

      ISO  year       country  break local_currency_code    F_local    F_usd  \
1     AFG  2014   Afghanistan  False                 AFN   5517.315   96.376   
3     AGO  2019        Angola   True                 AOA  58793.485  161.155   
31    ARM  2007       Armenia   True                 AMD  42240.351  123.481   
32    ARM  2010       Armenia  False                 AMD  59741.188  159.881   
35    ARM  2013       Armenia  False                 AMD  70296.765  171.612   
...   ...   ...           ...    ...                 ...        ...      ...   
1923  ZAF  2003  South Africa  False                 ZAR   2532.316  334.752   
1936  ZAF  2020  South Africa  False                 ZAR   6145.871  373.402   
1937  ZMB  2017        Zambia   True                 ZMW   4105.979  431.414   
1943  ZMB  2023        Zambia  False                 ZMW   3203.494  158.494   
1944  ZWE  2021      Zimbabwe  False                 ZWL  14481.879  163.540   

         M_local    M_usd    T_local   